### Prueba Factores de Elevación - Matrices de Fourness


In [2]:
import numpy as np

def furness(matrix, productions, attractions, max_iter=1000, tol=1e-5):
    """
    Ajuste de Furness (IPF) de una matriz origen-destino.
    
    Parámetros
    ----------
    matrix : np.ndarray
        Matriz O/D inicial (viajes).
    productions : np.ndarray
        Vector de subidos (producciones por origen).
    attractions : np.ndarray
        Vector de bajados (atracciones por destino).
    max_iter : int
        Número máximo de iteraciones.
    tol : float
        Tolerancia para convergencia.
    
    Retorna
    -------
    np.ndarray
        Matriz ajustada O/D.
    """
    T = matrix.astype(float)
    n, m = T.shape
    
    for it in range(max_iter):
        # Ajuste por filas (producciones)
        row_sums = T.sum(axis=1)
        for i in range(n):
            if row_sums[i] > 0:
                T[i, :] *= productions[i] / row_sums[i]
        
        # Ajuste por columnas (atracciones)
        col_sums = T.sum(axis=0)
        for j in range(m):
            if col_sums[j] > 0:
                T[:, j] *= attractions[j] / col_sums[j]
        
        # Comprobar convergencia
        if (np.allclose(T.sum(axis=1), productions, atol=tol) and 
            np.allclose(T.sum(axis=0), attractions, atol=tol)):
            break
    
    return T


def calcular_factores_elevacion(matrix_inicial, matrix_ajustada, productions, attractions):
    """
    Calcula factores de elevación y coeficiente de transbordo (simplificado).
    """
    subidos = productions.sum()
    bajados = attractions.sum()
    
    # Viajes iniciales y finales
    viajes_iniciales = matrix_inicial.sum()
    viajes_ajustados = matrix_ajustada.sum()
    
    Fe_subidos = subidos / viajes_iniciales
    Fe_bajados = bajados / viajes_iniciales
    
    # Para coeficiente de transbordo: estimamos como exceso de viajes frente a viajes directos
    CT = (viajes_ajustados - viajes_iniciales) / viajes_ajustados
    
    return {
        "Factor_elevacion_subidos": round(Fe_subidos, 3),
        "Factor_elevacion_bajados": round(Fe_bajados, 3),
        "Coef_transbordo": round(CT, 3)
    }


# Ejemplo de uso
if __name__ == "__main__":
    # Matriz inicial (3 zonas)
    matrix0 = np.array([
        [0, 100, 50],
        [80, 0, 40],
        [30, 60, 0]
    ])
    
    # Totales observados (subidos / bajados)
    productions = np.array([200, 150, 120])  # subidos
    attractions = np.array([180, 160, 130])  # bajados
    
    # Ajuste de Furness
    matrix_final = furness(matrix0, productions, attractions)
    
    # Factores
    factores = calcular_factores_elevacion(matrix0, matrix_final, productions, attractions)
    
    print("Matriz ajustada O/D:")
    print(np.round(matrix_final, 1))
    print("\nFactores:")
    print(factores)


Matriz ajustada O/D:
[[  0.  107.5  92.5]
 [112.5   0.   37.5]
 [ 67.5  52.5   0. ]]

Factores:
{'Factor_elevacion_subidos': 1.306, 'Factor_elevacion_bajados': 1.306, 'Coef_transbordo': 0.234}
